# Load Packages

In [1]:
import pandas as pd
import numpy as np
import pickle as pickle
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Import Data

In [2]:
train_set = pd.read_csv("../data/clean/twitter_training_clean.csv", index_col=0)
test_set = pd.read_csv("../data/clean/twitter_test_clean.csv", index_col=0)
val_set = pd.read_csv("../data/clean/twitter_validation_clean.csv", index_col=0)

X_train = train_set.drop(labels="Sentiment", axis=1)
y_train = train_set["Sentiment"]

X_test = test_set.drop(labels="Sentiment", axis=1)
y_test = test_set["Sentiment"]

X_val = val_set.drop(labels="Sentiment", axis=1)
y_val = val_set["Sentiment"]

## Encoding

In [3]:
encoder = OrdinalEncoder()
encoder = encoder.fit(X_train)
X_train = encoder.transform(X_train)

encoder = encoder.fit(X_val)
X_val = encoder.transform(X_val)

encoder = encoder.fit(X_test)
X_test = encoder.transform(X_test)

## Scaling

In [4]:
scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)

# Model

In [5]:
forest = RandomForestClassifier(random_state=42)
matthews_scorer = make_scorer(matthews_corrcoef)

## Hyperparameter Search

In [6]:
param_grid = {
    "n_estimators": [10, 20, 50, 100, 200, 500, 1000],
    "min_samples_split": [2, 3, 4]
}

grid_search = GridSearchCV(forest, param_grid, scoring=matthews_scorer, verbose=3)
grid_search.fit(X_train, y_train)

grid_search.cv_results_

Fitting 5 folds for each of 21 candidates, totalling 105 fits
[CV 1/5] END min_samples_split=2, n_estimators=10;, score=0.797 total time=   0.7s
[CV 2/5] END min_samples_split=2, n_estimators=10;, score=0.798 total time=   1.1s
[CV 3/5] END min_samples_split=2, n_estimators=10;, score=0.795 total time=   0.9s
[CV 4/5] END min_samples_split=2, n_estimators=10;, score=0.792 total time=   1.0s
[CV 5/5] END min_samples_split=2, n_estimators=10;, score=0.801 total time=   1.0s
[CV 1/5] END min_samples_split=2, n_estimators=20;, score=0.805 total time=   1.8s
[CV 2/5] END min_samples_split=2, n_estimators=20;, score=0.804 total time=   1.5s
[CV 3/5] END min_samples_split=2, n_estimators=20;, score=0.803 total time=   1.5s
[CV 4/5] END min_samples_split=2, n_estimators=20;, score=0.799 total time=   1.6s
[CV 5/5] END min_samples_split=2, n_estimators=20;, score=0.803 total time=   1.5s
[CV 1/5] END min_samples_split=2, n_estimators=50;, score=0.807 total time=   3.9s
[CV 2/5] END min_samples_

{'mean_fit_time': array([9.78763533e-01, 1.60986209e+00, 3.98717227e+00, 7.63885098e+00,
        2.16506672e+01, 5.73052060e+01, 9.36122429e+01, 9.85254622e-01,
        1.87200818e+00, 4.54722166e+00, 9.29013658e+00, 1.82149092e+01,
        4.64934238e+01, 9.16751994e+01, 9.43694448e-01, 1.91366458e+00,
        4.50526285e+00, 8.90195494e+00, 1.81664498e+01, 9.77182397e+03,
        2.18918578e+02]),
 'std_fit_time': array([1.00795449e-01, 8.98137322e-02, 3.02019145e-01, 1.00502998e-01,
        1.25142648e+01, 2.25914986e+01, 9.65750222e-01, 4.45289980e-02,
        3.46808350e-02, 6.82540039e-02, 1.28935875e-01, 3.35130234e-01,
        2.16030632e+00, 4.61335892e-01, 4.39257064e-02, 1.21251324e-01,
        1.69185198e-02, 1.46553622e-01, 3.75152699e-01, 1.45979496e+04,
        6.34190194e+00]),
 'mean_score_time': array([0.05239921, 0.0714838 , 0.1575242 , 0.28873196, 0.98272614,
        1.63238835, 3.11569977, 0.04752831, 0.08188968, 0.17468538,
        0.33289151, 0.62230191, 1.544911

In [7]:
best_model = grid_search.best_estimator_

## Training

In [8]:
train_score = best_model.score(X_train, y_train)
print("Training Score: ", train_score)

Training Score:  1.0


## Validation

In [9]:
val_score = best_model.score(X_val, y_val)
print("Validation Score: ", val_score)

Validation Score:  0.5956081081081082


## Test

In [10]:
test_score = best_model.score(X_test, y_test)
print("Test Score: ", test_score)

Test Score:  0.209
